# Post5G: hybrid deployment with OCUDU and OAI with Physical Radio Units in R2LAB and UERANSIM

## Prerequisites

Before starting, make sure you:

- Have completed the [OAI Physical RU in R2lab](./30_ru.ipynb) and [Add 5G components to a running network](./11_add_components.ipynb) tutorials.
- Have a basic understanding of Python.
- Are familiar with 5G architecture and 5G NR radio.

## Tutorial Overview

In this tutorial, you will deploy two real Radio Units (RUs) inside the [R2LAB anechoic chamber](https://r2lab.inria.fr/index.md) at SophiaNode: one using OpenAirInterface (OAI) and the other using OCUDU. You will also deploy a simulated radio network with UERANSIM. All components are served by the same 5G core in Open5GS.

For the sake of the example, we will attach the UERANSIM network after the deployment of the physical network.

### What You Will Build

The deployment consists of:
- **Core Network:** Open5GS core running on a virtual machine (VM).
- **RAN Deployment:** A split RAN architecture featuring:
  - A VM hosting the OAI CU-CP and CU-UP functions.
  - A dedicated bare-metal server (`sopnode-f1`) hosting the OAI DU and its 5G fronthaul connection.
  - A dedicated bare-metal server (`sopnode-f2`) hosting the OCUDU DU and its 5G fronthaul connection.
  - A VM hosting UERANSIM GNB and UEs.
- **Commercial RU:** An [AW2S Jaguar](https://www.aw2s.com/solutions/rrh-rru/) RU hosted inside the chamber, connected to the OAI DU over fiber.
- **SDR RU:** A [USRP N320](https://www.ettus.com/all-products/usrp-n320/) Software-Defined Radio (SDR) hosted inside the chamber, connected to the OCUDU DU over fiber.
- **Physical UEs:** Three physical 5G Quectel RM500Q-GL UEs (`qhat01`, `qhat02`, and `qhat03`), each attached to a Raspberry Pi 4 and connected to the radio networks.
- **Simulated UE:** Ten simulated UERANSIM UEs running on a didicated VM.

### Architecture & Infrastructure

This scenario builds upon the setup from the previous [RU tutorial](./30_ru.ipynb), adding a second bare-metal server to support the USRP N320 SDR radio:

| Component | Target Environment | Hardware / Machine |
| :--- | :--- | :--- |
| **5G Core** | Virtual Machine | SLICES-BI VM |
| **CU-CP & CU-UP** | Virtual Machine | SLICES-BI VM |
| **OAI DU & Fronthaul** | Bare-Metal | POS node (`sopnode-f1`) |
| **OCUDU DU & Fronthaul** | Bare-Metal | POS node (`sopnode-f2`) |
| **UERANSIM gNB and UEs** | Virtual Machine | SLICES-BI VM |

![RU topology](./figures/ru_full.png)

> **Note:** In this scenario, we connect an OCUDU DU with an OAI CU-CP and CU-UP setup.

## Metamodel and deployment model

### Metamodel

The metamodel follows the same pattern as in the other tutorials, we configure the USRP N320 for band `n78`, 20 MHz bandwidth, and 1x1 MIMO. The DU for this USRP uses the `DDDDDDDSUU` TDD pattern with a 5 ms period.

The TX and RX gains are tuned for the R2LAB environment. Examples of gNB configurations for R2LAB equipment are available [in this repository](https://github.com/sopnode/oai5g-rru/tree/main/rru).

In [ ]:
from metamodel.models.metamodel import MetaModel

metamodel=MetaModel.from_yaml(
"""
rus:
  jaguar:
    numberOfTxPorts: 2
    numberOfRxPorts: 2
    txAttenuationDB: 17

  n320:
    numberOfTxPorts: 1
    numberOfRxPorts: 1
    txGainDB: 35
    rxGainDB: 40

ran:
    components:
      cucp:
        type: cu_cp

      cuup:
        type: cu_up
        config:
          cu_cp: cucp

      du1:
        type: du
        config:
          radioConfiguration:
            nrBand: 78
            channelBandwidthMhz: 50
            pointAFrequencyArfcn: 626748
            ssbArfcn: 627360
            ssbPbchBlockPowerDbm: -10
            tddConfiguration:
              periodicityMilliseconds: 2.5
              nrofDownlinkSlots: 2
              nrofDownlinkSymbols: 6
              nrofUplinkSlots: 2
              nrofUplinkSymbols: 4
          ru: jaguar
          cu_cp: cucp
          cu_up: cuup

      du2:
        type: du
        config:
          radioConfiguration:
            nrBand: 78
            downlinkArfcn: 650000
            channelBandwidthMhz: 20
            prachConfigurationIndex: 1
            tddConfiguration:
              periodicityMilliseconds: 5
              nrofDownlinkSlots: 7
              nrofDownlinkSymbols: 6
              nrofUplinkSlots: 2
              nrofUplinkSymbols: 4
          ru: n320
          cu_cp: cucp
          cu_up: cuup
          

ues:
  qhat01:
    type: cots
    config:
      supi: "001010000000006"
      authenticationData:
        k: "fec86ba6eb707ed08905757b1bb44b8f"
        opc: "C42449363BBAD02B66D16BC975D77CC1"
      gnbs:
        - du1

  qhat02:
    type: cots
    config:
      supi: "001010000000007"
      authenticationData:
        k: "fec86ba6eb707ed08905757b1bb44b8f"
        opc: "C42449363BBAD02B66D16BC975D77CC1"
      gnbs:
        - du1

  qhat03:
    type: cots
    config:
      supi: "001010000000008"
      authenticationData:
        k: "fec86ba6eb707ed08905757b1bb44b8f"
        opc: "C42449363BBAD02B66D16BC975D77CC1"
      gnbs:
        - du2
""")

### Deployment Model

This tutorial builds on the multi-resource setup introduced in the previous scenario, combining virtual machines, bare-metal servers, RUs, and physical UEs.

#### Hardware Tuning & Network Configuration

* **USRP N320 Optimization:** The bare-metal fronthaul server `sopnode-f2` requires custom tuning specifically for the USRP N320 SDR. Key performance parameters—such as the sampling rate and master clock rate are configured directly through the node's provisioning settings.

For additional information on R2LAB hardware capabilities and network topologies, refer to the [SophiaNode Resources page](https://gitlab.inria.fr/slices-ri/resources/sites/sophianode/resources).

In [ ]:
from orchestrator.models.deployment import DeploymentModel

deployment_model = DeploymentModel.from_yaml(
"""
provisioners:
  bi-provisioner:
    type: bi
    params:
      infra: fr-sophia2-bi-vm1
      project: post5g-beta
      experiment: ru_tutorial_full
      create_experiment: true
      experiment_duration: 8h
      node_duration: 8h

  legacy_pos:
    type: legacy_pos
    params:
      attempt_reservation: false
      central_pos_ssh_info:
        hostname: duckburg.net.cit.tum.de
        port: 10022
        user: <REDACTED>

  r2lab:
    type: r2lab
    params:
      r2lab_slice: <REDACTED>
      reservation_params:
        email: <REDACTED>
        password: <REDACTED>

nodes:
  sopnode-f1:
    provisioner: legacy_pos
    provisioning_params:
      boot_params: >-
        biosdevname=0 net.ifnames=0 mitigations=off intel_iommu=on iommu=pt
        selinux=0 enforcing=0 isolcpus=managed_irq,16-63
        nohz_full=16-63 nohz=on rcu_nocbs=16-63
        kthread_cpus=0-4 irqaffinity=0-4 rcu_nocb_poll
        nosoftlockup intel_pstate=disable idle=poll skew_tick=1
        tsc=nowatchdog nmi_watchdog=0 softlockup_panic=0 audit=0 nosmt
    roles:
      - ran

  sopnode-f2:
    provisioner: legacy_pos
    provisioning_params:
      boot_params: >-
        biosdevname=0 net.ifnames=0 mitigations=off intel_iommu=on iommu=pt
        selinux=0 enforcing=0 isolcpus=managed_irq,16-63
        nohz_full=16-63 nohz=on rcu_nocbs=16-63
        kthread_cpus=0-4 irqaffinity=0-4 rcu_nocb_poll
        nosoftlockup intel_pstate=disable idle=poll skew_tick=1
        tsc=nowatchdog nmi_watchdog=0 softlockup_panic=0 audit=0 nosmt
    roles:
      - ran

  node-core:
    provisioner: bi-provisioner
    roles:
    - core

  node-ran:
    provisioner: bi-provisioner
    roles:
    - ran
    - cu_cp
    - cu_up

  node-ues:
    provisioner: bi-provisioner
    roles:
    - ue

rus:
  jaguar:
    vendor: aw2s
    model: jaguar
    provisioner: r2lab
  
  n320:
    vendor: usrp
    model: n320
    provisioner: r2lab

ues:
  qhat01:
    type: cots
    provisioner: r2lab

  qhat02:
    type: cots
    provisioner: r2lab

ran:
  implementation: oai
  components:
    cucp:
      type: cu_cp
      force_type_scheduling: true

    cuup:
      type: cu_up
      force_type_scheduling: true

    du1:
      type: du
      multus:
        enabled: true
        interfaces:
          - name: "ru"
            host_interface: "r2lab_aw2s"
            ip_address: "192.168.236.102"
      advanced_config:
        ssb_PositionsInBurst_Bitmap: 1
        num_tp_cores: 4
        rxfh_core_id: 12
        txfh_core_id: 14
        tp_cores: [16,18,20,22]
        pusch_TargetSNRx10: 290
        pucch_TargetSNRx10: 200
        initialDLBWPlocationAndBandwidth: 36300
        initialDLBWPcontrolResourceSetZero: 11
        initialULBWPlocationAndBandwidth: 36300
        ssb_perRACH_OccasionAndCB_PreamblesPerSSB: 15
        p0_NominalWithGrant: -76
        p0_nominal: -90
        useAdditionalOptions: "--thread-pool 17,19,21,23,25,27,29,31"
      node_name: sopnode-f1

    du2:
      implementation: ocudu
      type: du
      advanced_config:
        srate: 30.72
        master_clock_rate: "245.76e6"
      node_name: sopnode-f2

"""
)


## Provision the infrastructure

We can now provision the infrastructure nodes and kubernetes cluster.

In [3]:
from orchestrator.deployment import ProvisioningManagement, K8SManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)
await prov.provision_nodes(quiet=False)             

k8s = K8SManagement(preceding=prov)
await k8s.deploy(quiet=True)

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: No inventory was parsed, only implicit localhost is available

PLAY [localhost] ***************************************************************
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

TASK [provisioning/legacy_pos/provision : Register central POS host for 'sopnode-f1'] ***

PLAY [localhost] ***************************************************************

TASK [provisioning/bi/provision : Select project 'post5g-beta' for 'node-ran' provisioning] ***
changed: [localhost]

TASK [provisioning/legacy_pos/provi

In [4]:
from pathlib import Path
DIR="state/ru_full"
Path(DIR).mkdir(parents=True, exist_ok=True)

In [5]:
prov.metamodel.to_yaml_file(f"{DIR}/metamodel.prov.yaml")
prov.deployment_model.to_yaml_file(f"{DIR}/deployment_model.prov.yaml")

k8s.metamodel.to_yaml_file(f"{DIR}/metamodel.k8s.yaml")
k8s.deployment_model.to_yaml_file(f"{DIR}/deployment_model.k8s.yaml")

The infrastructure is now ready for the 5G network deployment.

## Deploy the 5G network

### Deploy the 5G core

With the infrastructure provisioned, start by deploying the 5G core.

In [6]:
from orchestrator.deployment import CoreManagement
core = CoreManagement(preceding=k8s)

In [7]:
await core.deploy()


PLAY [sopnode-f1] **************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-bsf' chart] ******

PLAY [sopnode-f1] **************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-amf' chart] ******

PLAY [sopnode-f1] **************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-udm' chart] ******

PLAY [sopnode-f1] **************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-nrf' chart] ******

PLAY [sopnode-f1] **************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-nssf' chart] *****

PLAY [sopnode-f1] **************************************************************

PLAY [sopnode-f1] **************************************************************

PLAY [sopnode-f

## Tune the RAN nodes

The DU needs additional CPU and network-interface tuning to achieve the performance required by the RU. These settings depend on the RU/DU pair, so they are applied directly to the machine with an Ansible playbook.

This tutorial uses a playbook tailored to this setup: [`tuning/ansible/playbooks/tuning_full.yaml`](tuning/ansible/playbooks/tuning_full.yaml).

In [8]:
from orchestrator.deployment import Miscellaneous

misc = MiscellaneousManagement(preceding=core)

In [9]:
await misc.run_playbook(
    playbook="./misc/ansible/playbooks/tuning_full.yaml",
    roles_paths=["./misc/ansible/roles"],
)


PLAY [Setup RAN nodes] *********************************************************

TASK [Gathering Facts] *********************************************************
[WARNING]: Host 'sopnode-f1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
ok: [sopnode-f1]
[WARNING]: Host 'sopnode-f2' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
ok: [sopnode-f2]

TASK [tuning/netplan : Upload netplan config to remote] ************************
changed: [sopnode-f2]
changed: [sopnode-f1]

TASK [tuning/netplan : Genera

### Deploy the 5G RAN

As the `sopnode-f1` and `sopnode-f2` machines are configured, we can provision the `jaguar` RU and `n320` USRP before deploying the DUs.

In [10]:
prov_rus = ProvisioningManagement(preceding=misc)
await prov_rus.provision()

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

TASK [Provision 'n320'] ********************************************************

TASK [Provision 'qhat01'] ******************************************************
included: provisioning/unified_radio/provision for localhost

TASK [Get access token if missing] *********************************************
included: provisioning/unified_radio/provision for localhost

TASK [Get access token if missing] **************************************

With the radio hardware provisioned, deploy the RAN components. The CU-CP and CU-UP will run on `node-ran`, and the DU will run on `sopnode-f1` and `sopnode-f2`.

In [11]:
from orchestrator.deployment import RANManagement
ran = RANManagement(preceding=prov_rus)
await ran.deploy()


PLAY [sopnode-f1] **************************************************************

PLAY [sopnode-f1] **************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-cucp' chart] *********

TASK [helm/prepare : Create Helm build directory for 'oai-cuup' chart] *********

PLAY [sopnode-f1] **************************************************************

PLAY [sopnode-f1] **************************************************************

TASK [helm/prepare : Create Helm build directory for 'ocudu-du2' chart] ********

TASK [helm/prepare : Create Helm build directory for 'oai-du1' chart] **********
[WARNING]: Host 'sopnode-f1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
changed: [sopnode-f1]

Check the pods in the `oai-ran` namespace to verify that the OAI RAN functions are running on the expected machines.

In [13]:
print((await prov_rus.run_node_command(
    command=["kubectl", "get", "pods", "-n", "oai-ran", "-o", "wide"],
    nodes=[prov_rus.deployment_model.params.k8s.master_node],
    become=True, quiet=True)))

NAME                        READY   STATUS    RESTARTS   AGE   IP              NODE         NOMINATED NODE   READINESS GATES
oai-cucp-699b847f9d-fqsrk   1/1     Running   0          53s   10.244.71.107   node-ran     <none>           <none>
oai-cuup-78d7ffb7cb-l628v   1/1     Running   0          43s   10.244.148.69   node-ran     <none>           <none>
oai-du1-8499db8d47-s8rk5    1/1     Running   0          36s   10.244.5.69     sopnode-f1   <none>           <none>


We can do the same for the OCUDU DU that is deployed in the `ocucu-ran` namespace.

In [14]:
print((await prov_rus.run_node_command(
    command=["kubectl", "get", "pods", "-n", "ocudu-ran", "-o", "wide"],
    nodes=[prov_rus.deployment_model.params.k8s.master_node],
    become=True, quiet=True)))

NAME                       READY   STATUS    RESTARTS   AGE   IP              NODE         NOMINATED NODE   READINESS GATES
ocudu-du2-6b59f4f4-hbkgm   1/1     Running   0          43s   10.244.83.214   sopnode-f2   <none>           <none>


Check the DUs logs to confirm that they are running.

In [15]:
logs = await ran.get_logs(quiet=True, flatten=False)

In [18]:
print ("DU1:")
print (logs["du1"]["du"])

DU1:
/proc/sys/kernel/core_pattern=core
No configuration file /opt/oai-gnb-aw2s/etc/gnb.conf found: attempting to find YAML config
== Configuration file:

# Active DUs
Active_gNBs:
  - du1
# Asn1_verbosity, choice in: none, info, annoying
Asn1_verbosity: none
gNBs:
  #////////// Identification parameters:
  - gNB_ID: 0x1
    gNB_DU_ID: 0x1
    gNB_name: du1
    tracking_area_code: 1
    plmn_list:
      - mcc: 001
        mnc: 01
        mnc_length: 2
        snssaiList:
          - sst: 1

    nr_cellid: 1
    ##### Physical parameters:
    min_rxtxtime: 6
    servingCellConfigCommon:
      #spCellConfigCommon
      - physCellId: 0
        #  downlinkConfigCommon
        #frequencyInfoDL
        # this is 3300.60 MHz + 53*12*30e-3 MHz = 3319.68
        absoluteFrequencySSB: 627360
        # this is 3300.60 MHz
        dl_absoluteFrequencyPointA: 626748
        #scs-SpecificCarrierList
        dl_offstToCarrier: 0
        # subcarrierSpacing
        # 0=kHz15, 1=kHz30, 2=kHz60, 3=kHz12

In [19]:
print ("DU2")
print (logs["du2"]["du"])

DU2

--== OCUDU DU (commit 920f88c) ==--

2026-10-26T15:21:51.296305 [GNB     ] [I] Built in Release mode using commit 920f88c on branch HEAD
2026-10-26T15:21:51.296805 [CONFIG  ] [I] DU input configuration (only non-default values): 
dryrun: false
gnb_id: 1
gnb_id_bit_length: 32
log:
  filename: stdout
  all_level: info
  config_level: info
metrics:
  autostart_stdout_metrics: true
f1ap:
  addrs: 10.244.71.107
  bind_addrs: 10.244.83.214
f1u:
  bind_port: 2153
  peer_port: 2153
  socket: "bind_addr: 10.244.83.214"
cell_cfg:
  pci: 0
  dl_arfcn: 650000
  band: 78
  common_scs: 30
  channel_bandwidth_MHz: 20
  nof_antennas_ul: 1
  nof_antennas_dl: 1
  plmn: 00101
  tac: 1
  slicing: "sst: 1"
  ssb:
    ssb_period: 20
  prach:
    prach_config_index: 1
  tdd_ul_dl_cfg:
    dl_ul_tx_period: 10
    nof_dl_slots: 7
    nof_dl_symbols: 6
    nof_ul_slots: 2
    nof_ul_symbols: 4
ru_sdr:
  srate: 30.72
  device_driver: uhd
  device_args: addr=192.168.235.105,clock_source=internal,time_source=

## Connect the UEs

We can now connect the UEs and verify that they are registered to the network and obtain their PDU sessions.

In [20]:
prov_ues = ProvisioningManagement(preceding=ran)
await prov_ues.provision()

In [22]:
from orchestrator.deployment import UEManagement
ues = UEManagement(preceding=prov_ues)
await ues.connect()

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

TASK [Connect 'qhat01'] ********************************************************
included: provisioning/unified_radio/ue/connect for localhost

TASK [Get access token if missing] *********************************************
included: provisioning/unified_radio/get_token for localhost

TASK [provisioning/unified_radio/get_token : Get 'r2lab' access token from unified radio managememt API] ***
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhos

In [25]:
logs = await ran.get_logs(quiet=True, flatten=False)

In [26]:
print ("DU1")
print (logs["du1"]["du"])

DU1
/proc/sys/kernel/core_pattern=core
No configuration file /opt/oai-gnb-aw2s/etc/gnb.conf found: attempting to find YAML config
== Configuration file:

# Active DUs
Active_gNBs:
  - du1
# Asn1_verbosity, choice in: none, info, annoying
Asn1_verbosity: none
gNBs:
  #////////// Identification parameters:
  - gNB_ID: 0x1
    gNB_DU_ID: 0x1
    gNB_name: du1
    tracking_area_code: 1
    plmn_list:
      - mcc: 001
        mnc: 01
        mnc_length: 2
        snssaiList:
          - sst: 1

    nr_cellid: 1
    ##### Physical parameters:
    min_rxtxtime: 6
    servingCellConfigCommon:
      #spCellConfigCommon
      - physCellId: 0
        #  downlinkConfigCommon
        #frequencyInfoDL
        # this is 3300.60 MHz + 53*12*30e-3 MHz = 3319.68
        absoluteFrequencySSB: 627360
        # this is 3300.60 MHz
        dl_absoluteFrequencyPointA: 626748
        #scs-SpecificCarrierList
        dl_offstToCarrier: 0
        # subcarrierSpacing
        # 0=kHz15, 1=kHz30, 2=kHz60, 3=kHz120

In [27]:
print ("DU2")
print (logs["du2"]["du"])

DU2

--== OCUDU DU (commit 920f88c) ==--

2026-10-26T15:21:51.296305 [GNB     ] [I] Built in Release mode using commit 920f88c on branch HEAD
2026-10-26T15:21:51.296805 [CONFIG  ] [I] DU input configuration (only non-default values): 
dryrun: false
gnb_id: 1
gnb_id_bit_length: 32
log:
  filename: stdout
  all_level: info
  config_level: info
metrics:
  autostart_stdout_metrics: true
f1ap:
  addrs: 10.244.71.107
  bind_addrs: 10.244.83.214
f1u:
  bind_port: 2153
  peer_port: 2153
  socket: "bind_addr: 10.244.83.214"
cell_cfg:
  pci: 0
  dl_arfcn: 650000
  band: 78
  common_scs: 30
  channel_bandwidth_MHz: 20
  nof_antennas_ul: 1
  nof_antennas_dl: 1
  plmn: 00101
  tac: 1
  slicing: "sst: 1"
  ssb:
    ssb_period: 20
  prach:
    prach_config_index: 1
  tdd_ul_dl_cfg:
    dl_ul_tx_period: 10
    nof_dl_slots: 7
    nof_dl_symbols: 6
    nof_ul_slots: 2
    nof_ul_symbols: 4
ru_sdr:
  srate: 30.72
  device_driver: uhd
  device_args: addr=192.168.235.105,clock_source=internal,time_source=

To test connectivity, send a ping from each UE over its radio interface.

In [28]:
prov_ues = ProvisioningManagement(preceding=ues)
command = ["ping", "-I", "wwan0", "-c", "10", "-D", "8.8.8.8"]
pings = await prov_ues.run_ue_command(command=command, ues=["qhat01", "qhat02", "qhat03"], quiet=True, flatten=False)

Ping results from `qhat01`:

In [29]:
print (pings["qhat01"])

PING 8.8.8.8 (8.8.8.8) from 11.1.1.5 wwan0: 56(84) bytes of data.
[1793028455.394623] 64 bytes from 8.8.8.8: icmp_seq=1 ttl=114 time=29.8 ms
[1793028456.399535] 64 bytes from 8.8.8.8: icmp_seq=2 ttl=114 time=32.8 ms
[1793028457.399428] 64 bytes from 8.8.8.8: icmp_seq=3 ttl=114 time=30.8 ms
[1793028458.404327] 64 bytes from 8.8.8.8: icmp_seq=4 ttl=114 time=33.8 ms
[1793028459.404329] 64 bytes from 8.8.8.8: icmp_seq=5 ttl=114 time=32.0 ms
[1793028460.405241] 64 bytes from 8.8.8.8: icmp_seq=6 ttl=114 time=31.9 ms
[1793028461.409160] 64 bytes from 8.8.8.8: icmp_seq=7 ttl=114 time=33.8 ms
[1793028462.409555] 64 bytes from 8.8.8.8: icmp_seq=8 ttl=114 time=32.3 ms
[1793028463.409457] 64 bytes from 8.8.8.8: icmp_seq=9 ttl=114 time=30.8 ms
[1793028464.414365] 64 bytes from 8.8.8.8: icmp_seq=10 ttl=114 time=33.8 ms

--- 8.8.8.8 ping statistics ---
10 packets transmitted, 10 received, 0% packet loss, time 9016ms
rtt min/avg/max/mdev = 29.845/32.200/33.848/1.340 ms


Ping results from `qhat02`:

In [30]:
print (pings["qhat02"])

PING 8.8.8.8 (8.8.8.8) from 11.1.1.4 wwan0: 56(84) bytes of data.
[1793028455.824374] 64 bytes from 8.8.8.8: icmp_seq=1 ttl=114 time=34.4 ms
[1793028456.816257] 64 bytes from 8.8.8.8: icmp_seq=2 ttl=114 time=24.8 ms
[1793028457.823764] 64 bytes from 8.8.8.8: icmp_seq=3 ttl=114 time=30.4 ms
[1793028458.829178] 64 bytes from 8.8.8.8: icmp_seq=4 ttl=114 time=34.4 ms
[1793028459.829090] 64 bytes from 8.8.8.8: icmp_seq=5 ttl=114 time=32.9 ms
[1793028460.829000] 64 bytes from 8.8.8.8: icmp_seq=6 ttl=114 time=30.9 ms
[1793028461.833913] 64 bytes from 8.8.8.8: icmp_seq=7 ttl=114 time=33.9 ms
[1793028462.833818] 64 bytes from 8.8.8.8: icmp_seq=8 ttl=114 time=31.8 ms
[1793028463.833832] 64 bytes from 8.8.8.8: icmp_seq=9 ttl=114 time=30.0 ms
[1793028464.839246] 64 bytes from 8.8.8.8: icmp_seq=10 ttl=114 time=34.4 ms

--- 8.8.8.8 ping statistics ---
10 packets transmitted, 10 received, 0% packet loss, time 9015ms
rtt min/avg/max/mdev = 24.820/31.770/34.358/2.825 ms


Ping results from `qhat03`:

In [31]:
print (pings["qhat03"])

PING 8.8.8.8 (8.8.8.8) from 11.1.1.6 wwan0: 56(84) bytes of data.
[1793028460.172774] 64 bytes from 8.8.8.8: icmp_seq=1 ttl=114 time=32.2 ms
[1793028461.165160] 64 bytes from 8.8.8.8: icmp_seq=2 ttl=114 time=23.2 ms
[1793028462.177556] 64 bytes from 8.8.8.8: icmp_seq=3 ttl=114 time=34.2 ms
[1793028463.177464] 64 bytes from 8.8.8.8: icmp_seq=4 ttl=114 time=32.7 ms
[1793028464.177864] 64 bytes from 8.8.8.8: icmp_seq=5 ttl=114 time=31.2 ms
[1793028465.182764] 64 bytes from 8.8.8.8: icmp_seq=6 ttl=114 time=34.7 ms
[1793028466.182760] 64 bytes from 8.8.8.8: icmp_seq=7 ttl=114 time=32.8 ms
[1793028467.182672] 64 bytes from 8.8.8.8: icmp_seq=8 ttl=114 time=30.7 ms
[1793028468.180078] 64 bytes from 8.8.8.8: icmp_seq=9 ttl=114 time=26.2 ms
[1793028469.187485] 64 bytes from 8.8.8.8: icmp_seq=10 ttl=114 time=32.2 ms

--- 8.8.8.8 ping statistics ---
10 packets transmitted, 10 received, 0% packet loss, time 9015ms
rtt min/avg/max/mdev = 23.169/31.017/34.716/3.429 ms


Finally, save the metamodel and deployment model to keep a record of this experiment.

In [32]:
ues.metamodel.to_yaml_file(f"{DIR}/metamodel.ues.yaml")
ues.deployment_model.to_yaml_file(f"{DIR}/deployment_model.ues.yaml")

## Add component on the fly

In [33]:
from metamodel.models.ru import RU

metamodel, deployment_model = prov_ues.output()

metamodel.rus["sim_ru"] = RU()

In [35]:
import yaml
merged_model = yaml.safe_load(
"""
type: gnb
implementation: ueransim
""")

ran = RANManagement(metamodel=metamodel, deployment_model=deployment_model)

ran.add_component(name="gnb", merged=merged_model)
await ran.deploy()


PLAY [sopnode-f1] **************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-cuup' chart] *********

PLAY [sopnode-f1] **************************************************************

TASK [helm/prepare : Create Helm build directory for 'ocudu-du2' chart] ********

PLAY [sopnode-f1] **************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-cucp' chart] *********

PLAY [sopnode-f1] **************************************************************

TASK [helm/prepare : Create Helm build directory for 'ueransim-gnb' chart] *****

PLAY [sopnode-f1] **************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-du1' chart] **********
[WARNING]: Host 'sopnode-f1' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different inte

In [38]:
prov = ProvisioningManagement(preceding=ran)

print((await prov.run_node_command(
    command=["kubectl", "get", "pods", "-n", "ueransim-ran", "-o", "wide"],
    nodes=["node-ran"],
    become=True, quiet=True)))

NAME                            READY   STATUS    RESTARTS   AGE   IP               NODE         NOMINATED NODE   READINESS GATES
ueransim-gnb-5dfb6cff94-mdlhm   1/1     Running   0          26s   10.244.157.186   sopnode-f2   <none>           <none>


In [39]:
print((await ran.get_logs(components=["gnb"], quiet=True)))

UERANSIM v3.3.0
[2026-10-26 15:28:57.087] [sctp] [info] Trying to establish SCTP connection... (10.244.134.4:38412)
[2026-10-26 15:28:57.090] [sctp] [info] SCTP connection established (10.244.134.4:38412)
[2026-10-26 15:28:57.090] [sctp] [debug] SCTP association setup ascId[4]
[2026-10-26 15:28:57.090] [ngap] [debug] Sending NG Setup Request
[2026-10-26 15:28:57.104] [ngap] [debug] NG Setup Response received
[2026-10-26 15:28:57.104] [ngap] [info] NG Setup procedure is successful



In [40]:
from orchestrator.deployment import UEManagement
ues = UEManagement(preceding=ran)

In [42]:
batch = 1
ue_merged_models = []
ue_names = []


for i in range(1,11):
    name = f"sim_ue{i:04d}"
    merged_model = yaml.safe_load(
    f"""
    type: containerized
    implementation: ueransim
    config:
        supi: '001010000{batch:02d}{i:04d}'
        gnbs:
        - gnb
    """
    )
    ue_names.append(name)
    ue_merged_models.append(merged_model)

for name, merged_model in list(zip(ue_names, ue_merged_models)):
    ues.add_component(name=name, merged=merged_model)

In [43]:
core = CoreManagement(preceding=ues)
await core.register_ues(ues=ue_names)


PLAY [sopnode-f1] **************************************************************

TASK [database/subscriber/add/open5gs : Add UE 'sim_ue0001' to Open5GS database] ***

PLAY [sopnode-f1] **************************************************************

TASK [database/subscriber/add/open5gs : Add UE 'sim_ue0005' to Open5GS database] ***

PLAY [sopnode-f1] **************************************************************

TASK [database/subscriber/add/open5gs : Add UE 'sim_ue0007' to Open5GS database] ***

PLAY [sopnode-f1] **************************************************************

TASK [database/subscriber/add/open5gs : Add UE 'sim_ue0010' to Open5GS database] ***

PLAY [sopnode-f1] **************************************************************

TASK [database/subscriber/add/open5gs : Add UE 'sim_ue0003' to Open5GS database] ***

PLAY [sopnode-f1] **************************************************************

TASK [database/subscriber/add/open5gs : Add UE 'sim_ue0004' to Open5GS databa

In [44]:
ues = UEManagement(preceding=core)
await ues.deploy(components=ue_names)


PLAY [sopnode-f1] **************************************************************

TASK [helm/prepare : Create Helm build directory for 'ueransim-sim-ue0001' chart] ***

PLAY [sopnode-f1] **************************************************************

TASK [helm/prepare : Create Helm build directory for 'ueransim-sim-ue0006' chart] ***

PLAY [sopnode-f1] **************************************************************

PLAY [sopnode-f1] **************************************************************

TASK [helm/prepare : Create Helm build directory for 'ueransim-sim-ue0005' chart] ***

PLAY [sopnode-f1] **************************************************************

TASK [helm/prepare : Create Helm build directory for 'ueransim-sim-ue0004' chart] ***

PLAY [sopnode-f1] **************************************************************

TASK [helm/prepare : Create Helm build directory for 'ueransim-sim-ue0002' chart] ***

TASK [helm/prepare : Create Helm build directory for 'ueransim-sim-ue000

In [45]:
prov = ProvisioningManagement(preceding=ran)

print((await prov.run_node_command(
    command=["kubectl", "get", "pods", "-n", "ueransim-ues", "-o", "wide"],
    nodes=["node-ran"],
    become=True, quiet=True)))

In [54]:
command = ["ip", "addr", "show", "dev", "wwan0"]
ips = await prov_ues.run_ue_command(command=command, ues=["qhat01", "qhat02", "qhat03"], quiet=True, flatten=False)


In [55]:
def get_ip(output):
    import re
    match = re.search(r"\binet\s+(\d{1,3}(?:\.\d{1,3}){3})/", output)
    ip_address = match.group(1) if match else None
    return ip_address


In [64]:
command = ["ping", "-I", "uesimtun0", "-c", 2, "-q", get_ip(ips["qhat01"])]
pings = await ues.run_command(command=command, components=ue_names, quiet=True, flatten=False)

In [80]:
for ue, ping in pings.items():
    head = f"{ue} qhat01:"
    tab = " "*(len(head) + 1)
    data = f"\n{tab}".join(ping.splitlines()[-2:])
    print (f"{head} {data}")

sim_ue0001 qhat01: 2 packets transmitted, 2 received, 0% packet loss, time 1001ms
                   rtt min/avg/max/mdev = 11.628/21.502/31.376/9.874 ms
sim_ue0002 qhat01: 2 packets transmitted, 2 received, 0% packet loss, time 1001ms
                   rtt min/avg/max/mdev = 26.418/26.624/26.830/0.206 ms
sim_ue0003 qhat01: 2 packets transmitted, 2 received, 0% packet loss, time 1002ms
                   rtt min/avg/max/mdev = 25.356/26.104/26.852/0.748 ms
sim_ue0004 qhat01: 2 packets transmitted, 2 received, 0% packet loss, time 1002ms
                   rtt min/avg/max/mdev = 27.827/27.957/28.087/0.130 ms
sim_ue0005 qhat01: 2 packets transmitted, 2 received, 0% packet loss, time 1000ms
                   rtt min/avg/max/mdev = 25.052/25.057/25.062/0.005 ms
sim_ue0006 qhat01: 2 packets transmitted, 2 received, 0% packet loss, time 1000ms
                   rtt min/avg/max/mdev = 28.076/28.193/28.310/0.117 ms
sim_ue0007 qhat01: 2 packets transmitted, 2 received, 0% packet loss, time 1

In [81]:
command = ["ping", "-I", "uesimtun0", "-c", 2, "-q", get_ip(ips["qhat02"])]
pings = await ues.run_command(command=command, components=ue_names, quiet=True, flatten=False)

In [82]:
for ue, ping in pings.items():
    head = f"{ue} qhat02:"
    tab = " "*(len(head) + 1)
    data = f"\n{tab}".join(ping.splitlines()[-2:])
    print (f"{head} {data}")

sim_ue0001 qhat02: 2 packets transmitted, 2 received, 0% packet loss, time 1002ms
                   rtt min/avg/max/mdev = 28.150/28.847/29.544/0.697 ms
sim_ue0002 qhat02: 2 packets transmitted, 2 received, 0% packet loss, time 1002ms
                   rtt min/avg/max/mdev = 19.931/20.797/21.663/0.866 ms
sim_ue0003 qhat02: 2 packets transmitted, 2 received, 0% packet loss, time 1002ms
                   rtt min/avg/max/mdev = 25.874/27.480/29.086/1.606 ms
sim_ue0004 qhat02: 2 packets transmitted, 2 received, 0% packet loss, time 1001ms
                   rtt min/avg/max/mdev = 29.199/29.749/30.299/0.550 ms
sim_ue0005 qhat02: 2 packets transmitted, 2 received, 0% packet loss, time 1001ms
                   rtt min/avg/max/mdev = 29.258/29.799/30.340/0.541 ms
sim_ue0006 qhat02: 2 packets transmitted, 2 received, 0% packet loss, time 1002ms
                   rtt min/avg/max/mdev = 26.647/27.676/28.706/1.029 ms
sim_ue0007 qhat02: 2 packets transmitted, 2 received, 0% packet loss, time 1

In [83]:
command = ["ping", "-I", "uesimtun0", "-c", 2, "-q", get_ip(ips["qhat03"])]
pings = await ues.run_command(command=command, components=ue_names, quiet=True, flatten=False)

In [84]:
for ue, ping in pings.items():
    head = f"{ue} qhat03:"
    tab = " "*(len(head) + 1)
    data = f"\n{tab}".join(ping.splitlines()[-2:])
    print (f"{head} {data}")

sim_ue0001 qhat03: 2 packets transmitted, 2 received, 0% packet loss, time 1001ms
                   rtt min/avg/max/mdev = 13.795/13.838/13.882/0.043 ms
sim_ue0002 qhat03: 2 packets transmitted, 2 received, 0% packet loss, time 1002ms
                   rtt min/avg/max/mdev = 26.514/27.183/27.852/0.669 ms
sim_ue0003 qhat03: 2 packets transmitted, 2 received, 0% packet loss, time 1002ms
                   rtt min/avg/max/mdev = 25.872/26.306/26.740/0.434 ms
sim_ue0004 qhat03: 2 packets transmitted, 2 received, 0% packet loss, time 1002ms
                   rtt min/avg/max/mdev = 24.687/26.317/27.948/1.630 ms
sim_ue0005 qhat03: 2 packets transmitted, 2 received, 0% packet loss, time 1001ms
                   rtt min/avg/max/mdev = 21.370/21.843/22.316/0.473 ms
sim_ue0006 qhat03: 2 packets transmitted, 2 received, 0% packet loss, time 1001ms
                   rtt min/avg/max/mdev = 19.981/21.556/23.131/1.575 ms
sim_ue0007 qhat03: 2 packets transmitted, 2 received, 0% packet loss, time 1